# Power Analysis & Sample Size Calculation

This notebook demonstrates how to use the `PowerAnalyzer` class to determine the required sample size for an A/B experiment before it launches.

**Key concepts:**
- **Statistical Power**: The probability of detecting a true effect (typically 80%)
- **Minimum Detectable Effect (MDE)**: The smallest improvement we want to reliably detect
- **Significance Level (alpha)**: The false positive rate we tolerate (typically 5%)

A well-powered experiment avoids two failure modes:
1. Running too short and missing real effects (underpowered)
2. Running too long and wasting traffic (overpowered)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.power_analysis import PowerAnalyzer

analyzer = PowerAnalyzer()
print("PowerAnalyzer loaded successfully.")

## Sample Size for Different MDE Values

Let's calculate the required sample size per variant for a baseline conversion rate of 5%, across a range of minimum detectable effects (absolute). Smaller effects require exponentially larger samples.

In [ ]:
baseline_rate = 0.05
mde_values = [0.005, 0.01, 0.015, 0.02, 0.025, 0.03]

print(f"Baseline conversion rate: {baseline_rate*100:.1f}%")
print(f"{'MDE (abs)':>12} {'MDE (rel)':>12} {'Sample/variant':>16} {'Total sample':>14}")
print("-" * 58)

for mde in mde_values:
    n = analyzer.required_sample_size(
        baseline_rate=baseline_rate,
        minimum_detectable_effect=mde,
        alpha=0.05,
        power=0.80
    )
    relative_mde = mde / baseline_rate * 100
    print(f"{mde:>12.3f} {relative_mde:>10.1f}% {n:>16,} {2*n:>14,}")

## Impact of Power Level on Sample Size

Higher power (e.g., 90% vs. 80%) gives more confidence that we will detect the effect, but costs more samples.

In [ ]:
power_levels = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
mde_fixed = 0.01

print(f"Baseline: {baseline_rate*100:.1f}%, MDE: {mde_fixed*100:.2f}% (absolute)")
print(f"{'Power':>8} {'N per variant':>16} {'Increase vs 80%':>18}")
print("-" * 46)

n_at_80 = analyzer.required_sample_size(baseline_rate, mde_fixed, power=0.80)

for pwr in power_levels:
    n = analyzer.required_sample_size(baseline_rate, mde_fixed, power=pwr)
    increase = (n - n_at_80) / n_at_80 * 100
    print(f"{pwr:>8.0%} {n:>16,} {increase:>+16.1f}%")

## Power Curves Visualization

A power curve shows how statistical power increases with sample size for a fixed MDE. We plot curves for several MDE values to illustrate the tradeoff.

In [ ]:
sample_sizes = list(range(500, 50001, 500))
mde_for_curves = [0.005, 0.01, 0.02, 0.03]

fig, ax = plt.subplots(figsize=(10, 6))

for mde in mde_for_curves:
    curve_data = analyzer.power_curve(
        baseline_rate=baseline_rate,
        sample_sizes=sample_sizes,
        mde=mde
    )
    powers = [d['power'] for d in curve_data]
    relative_mde = mde / baseline_rate * 100
    ax.plot(sample_sizes, powers, linewidth=2, label=f'MDE = {mde:.3f} ({relative_mde:.0f}% relative)')

ax.axhline(y=0.80, color='gray', linestyle='--', alpha=0.7, label='80% power threshold')
ax.set_xlabel('Sample Size per Variant', fontsize=12)
ax.set_ylabel('Statistical Power', fontsize=12)
ax.set_title('Power Curves: How Sample Size Affects Detection Probability', fontsize=14)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 50000)
plt.tight_layout()
plt.show()

## Experiment Duration Estimation

Given daily traffic and the required sample size, we can estimate how many days the experiment will need to run.

In [ ]:
required_n = analyzer.required_sample_size(
    baseline_rate=0.05,
    minimum_detectable_effect=0.01,
    power=0.80
)

result = analyzer.estimate_duration(
    required_sample_size=required_n,
    daily_traffic=10000,
    allocation_pct=1.0,
    num_variants=2
)

print(f"Required sample size per variant: {result.sample_size_per_variant:,}")
print(f"Total sample size (both variants): {result.total_sample_size:,}")
print(f"Estimated duration: {result.estimated_duration_days} days")
print(f"\nWith 10,000 daily visitors and full allocation:")
print(f"  Each variant gets ~5,000 users/day")
print(f"  Experiment needs ~{result.estimated_duration_days} days to reach power")

## Sensitivity Analysis

If you already have a fixed sample size (e.g., due to a limited experiment window), sensitivity analysis tells you the minimum effect you can detect.

In [ ]:
fixed_samples = [5000, 10000, 20000, 50000]

print(f"Baseline rate: {baseline_rate*100:.1f}% | Power: 80% | Alpha: 5%")
print(f"{'Sample Size':>14} {'MDE (abs)':>12} {'MDE (rel)':>12} {'Detectable Rate':>18}")
print("-" * 60)

for n in fixed_samples:
    result = analyzer.sensitivity_analysis(
        baseline_rate=baseline_rate,
        sample_size=n
    )
    print(f"{n:>14,} {result['minimum_detectable_effect']:>12.4f} "
          f"{result['relative_mde']:>10.1f}% {result['detectable_rate']:>18.4f}")